In [4]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers, models

# 1. Load the dataset
# We split it: 80% for training, 20% for validation
(train_ds, val_ds), ds_info = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:]'],
    as_supervised=True,  # Returns (image, label) tuples
    with_info=True
)

# 2. Preprocessing Function
def preprocess_image(image, label):
    image = tf.image.resize(image, (150, 150)) # Resize to standard size
    image = tf.cast(image, tf.float32) / 255.0  # Normalize to [0, 1]
    return image, label

# 3. Build the Data Pipeline
# Shuffle, Batch, and Autotune for performance
BATCH_SIZE = 32
train_ds = train_ds.map(preprocess_image).shuffle(1000).batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess_image).batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)

# 4. Build the CNN Model
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    layers.MaxPooling2D(2, 2),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    
    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.Dense(1, activation='sigmoid') # Binary classification
])

# 5. Compile and Train
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

print("Training starting...")
model.fit(train_ds, epochs=10, validation_data=val_ds)

# 6. Save the model
model.save('cats_vs_dogs_tfds.h5')
print("Model saved successfully.")

ModuleNotFoundError: No module named 'importlib_resources'

In [5]:
pip install importlib_resources

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\Laksh\Desktop\TFexp\TF\Scripts\python.exe -m pip install --upgrade pip


In [8]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

# 1. Load the built-in CIFAR-10 dataset
print("Loading CIFAR-10 dataset...")
(x_train_all, y_train_all), (x_test_all, y_test_all) = tf.keras.datasets.cifar10.load_data()

# 2. Filter for Cats (label 3) and Dogs (label 5)
# This creates a boolean mask to pick only the images we want
train_mask = np.isin(y_train_all, [3, 5]).flatten()
test_mask = np.isin(y_test_all, [3, 5]).flatten()

x_train, y_train = x_train_all[train_mask], y_train_all[train_mask]
x_test, y_test = x_test_all[test_mask], y_test_all[test_mask]

# 3. Preprocess the data
# Map Cat(3) to 0 and Dog(5) to 1
y_train = np.where(y_train == 3, 0, 1)
y_test = np.where(y_test == 3, 0, 1)

# Normalize pixel values to [0, 1]
x_train, x_test = x_train / 255.0, x_test / 255.0

print(f"Training set size: {len(x_train)} images")
print(f"Test set size: {len(x_test)} images")

# 4. Build the CNN Model
model = models.Sequential([
    # CIFAR-10 images are 32x32 pixels with 3 color channels (RGB)
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid') # Sigmoid for binary output
])

# 5. Compile and Train
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

print("\nStarting Training...")
model.fit(x_train, y_train, epochs=10, validation_data=(x_test, y_test))

# 6. Evaluate
loss, acc = model.evaluate(x_test, y_test)
print(f"\nFinal Accuracy: {acc*100:.2f}%")

Loading CIFAR-10 dataset...
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 19s 0us/step


C:\Users\Laksh\Desktop\TFexp\TF\Lib\site-packages\keras\src\datasets\cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


Training set size: 10000 images
Test set size: 2000 images


C:\Users\\Desktop\TFexp\TF\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Starting Training...
Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 18ms/step - accuracy: 0.5740 - loss: 0.6726 - val_accuracy: 0.6350 - val_loss: 0.6356
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.6618 - loss: 0.6180 - val_accuracy: 0.6780 - val_loss: 0.5847
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.7050 - loss: 0.5693 - val_accuracy: 0.6985 - val_loss: 0.5710
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.7301 - loss: 0.5371 - val_accuracy: 0.7315 - val_loss: 0.5231
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.7525 - loss: 0.5057 - val_accuracy: 0.7510 - val_loss: 0.5081
Epoch 6/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.7642 - loss: 0.4777 - val_accuracy: 0.7495 - val_loss: 0.5009
Epoch 7/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.7794 - loss: 0.4518 - val_accuracy: 0.7520 - val_loss: 0.4915
Epoch 8/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.8006 - l

In [10]:
import numpy as np
from tensorflow.keras.preprocessing import image

def predict_animal(img_path):
    # 1. Load the image and resize it to 32x32 (CIFAR-10 size)
    img = image.load_img(img_path, target_size=(32, 32))
    
    # 2. Convert image to array and normalize
    img_array = image.img_to_array(img)
    img_array = img_array / 255.0  # Same normalization as training
    
    # 3. Add a fourth dimension (batch size) -> (1, 32, 32, 3)
    img_array = np.expand_dims(img_array, axis=0)
    
    # 4. Make the prediction
    prediction = model.predict(img_array)
    
    # 5. Interpret the result
    # Remember: Cat was 0, Dog was 1
    if prediction[0][0] > 0.5:
        print(f"Confidence: {prediction[0][0]*100:.2f}% -> It's a DOG! 🐶")
    else:
        print(f"Confidence: {(1-prediction[0][0])*100:.2f}% -> It's a CAT! 🐱")

# Usage: 
# Replace 'my_pet.jpg' with the actual path to the image you downloaded
# predict_animal('my_pet.jpg')

In [12]:
predict_animal(r"C:\Users\Laksh\Downloads\images.jpg")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 186ms/step
Confidence: 79.98% -> It's a CAT! 🐱
